# PI4 — Diferenças por rede e zona

Notebook preparatório da etapa **E03**. Compara os indicadores de infraestrutura de Guaratinguetá por dependência administrativa e localização urbana/rural. Bases com menos de 5 escolas válidas são sinalizadas para evitar conclusões frágeis.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
from datetime import datetime
import subprocess, sys
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

PROJECT_NAME='UNIVESP — PI4 — Infraestrutura Escolar — Guaratinguetá'
candidates=[Path('/content/drive/MyDrive')/PROJECT_NAME,Path('/content/drive/My Drive')/PROJECT_NAME]
PROJECT_ROOT=next((p for p in candidates if p.exists()),None)
if PROJECT_ROOT is None: raise FileNotFoundError('Adicione um atalho da pasta compartilhada do PI4 ao Meu Drive.')
BASE_ANALITICA=PROJECT_ROOT/'01_Dados'/'2_tratamentos_dados'/'base_analitica'


In [ ]:
!rm -rf /content/pi4_repo
!git clone -q --depth 1 https://github.com/felipecsr/univesp-projeto-integrador-4.git /content/pi4_repo
sys.path.insert(0,'/content/pi4_repo')
from src.validate_analytic_base import validate_execution
from src.eda import latest_execution_dir,load_materialized_panel,infrastructure_by_cut

REPO_COMMIT=subprocess.check_output(['git','-C','/content/pi4_repo','rev-parse','HEAD'],text=True).strip()
RUN_DIR=latest_execution_dir(BASE_ANALITICA)
reconciliacao=validate_execution(RUN_DIR)
display(reconciliacao)
assert not (reconciliacao['status']=='FAIL').any(),'P04 falhou.'
print('GATE P04: PASS')


In [ ]:
panel=load_materialized_panel(RUN_DIR)
por_rede=infrastructure_by_cut(panel,'TP_DEPENDENCIA')
por_zona=infrastructure_by_cut(panel,'TP_LOCALIZACAO')

display(por_rede[por_rede['ano']=='2025'])
display(por_zona[por_zona['ano']=='2025'])


## Bases pequenas

A rede federal e o recorte rural podem ter poucas escolas. O indicador `base_pequena=True` significa que o percentual deve ser tratado como descritivo daquele pequeno conjunto, não como evidência robusta de diferença entre grupos.


In [ ]:
bases_pequenas=pd.concat([por_rede,por_zona],ignore_index=True)
bases_pequenas=bases_pequenas[bases_pequenas['base_pequena']].copy()
display(bases_pequenas[['ano','corte','grupo','indicador','escolas_validas','pct_escolas_com_item']])


## Visualização de 2025 por rede


In [ ]:
rede_2025=por_rede[por_rede['ano']=='2025'].copy()
pivot=rede_2025.pivot(index='indicador',columns='grupo',values='pct_escolas_com_item')*100
display(pivot)

fig,ax=plt.subplots(figsize=(10,6))
pivot.plot(kind='bar',ax=ax)
ax.set_title('Guaratinguetá — infraestrutura por rede em 2025')
ax.set_ylabel('% de escolas com o item')
ax.set_xlabel('Indicador')
ax.legend(title='Rede')
ax.grid(axis='y',alpha=0.2)
plt.xticks(rotation=70,ha='right')
plt.tight_layout()
plt.show()


## Materialização


In [ ]:
EDA_ROOT=PROJECT_ROOT/'01_Dados'/'2_tratamentos_dados'/'eda'
EDA_RUN=EDA_ROOT/f'rede_zona_{datetime.now().strftime("%Y%m%d_%H%M%S")}'
EDA_RUN.mkdir(parents=True,exist_ok=True)
por_rede.to_csv(EDA_RUN/'infraestrutura_por_rede.csv',index=False,encoding='utf-8')
por_zona.to_csv(EDA_RUN/'infraestrutura_por_zona.csv',index=False,encoding='utf-8')
bases_pequenas.to_csv(EDA_RUN/'bases_pequenas.csv',index=False,encoding='utf-8')
reconciliacao.to_csv(EDA_RUN/'gate_p04.csv',index=False,encoding='utf-8')
manifest=pd.DataFrame([{'executado_em':datetime.now().isoformat(timespec='seconds'),'repo_commit':REPO_COMMIT,'base_origem':str(RUN_DIR)}])
manifest.to_csv(EDA_RUN/'manifesto_rede_zona.csv',index=False,encoding='utf-8')
display(manifest)
print('Resultados gravados em:',EDA_RUN)


## Evidências manuais de E03

Guardar: **(1)** gate P04 PASS; **(2)** tabela 2025 por rede; **(3)** tabela de bases pequenas; **(4)** gráfico por rede. Depois, deixar `E03M` em `Revisão`.
